In [1]:
# !mkdir -p mjd_data
# !mkdir -p mjd_features/train
# !mkdir -p mjd_features/test

# %cd /home/nbatjargal/private/capstone_project
%cd /Users/nomin1/Desktop/capstone

/Users/nomin1/Desktop/capstone


/Users/nomin1/Library/Python/3.12/lib/python/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
# !test -d Majorana-Neutrino-Hunt || git clone https://github.com/YooNice100/Majorana-Neutrino-Hunt.git

In [4]:
import sys
import os
import numpy as np
import pandas as pd
import h5py
import glob
from pathlib import Path
from functools import reduce

sys.path.append("Majorana-Neutrino-Hunt")

from src.parameters.frequency_domain import (
    compute_spectral_centroid_power,
    compute_total_power
)
from src.parameters.gradient_features import (
    compute_current_kurtosis,
    compute_current_skewness,
    compute_current_width
)
from src.parameters.tail_features import compute_tail_slope
from src.parameters.time_domain import compute_time_to_peak, compute_time_to_main_peak
from src.utils.io import load_hdf5

In [5]:
os.getcwd()

'/Users/nomin1/Desktop/capstone'

In [5]:
def download_mjd_files(split_name, file_indices):
    os.makedirs("mjd_data", exist_ok=True)

    for file_idx in file_indices:
        if split_name == "train":
            hdf5_file = f"MJD_Train_{file_idx}.hdf5"
        elif split_name == "test":
            hdf5_file = f"MJD_Test_{file_idx}.hdf5"
        else:
            raise ValueError("split_name must be 'train' or 'test'")

        url = f"https://zenodo.org/records/8257027/files/{hdf5_file}"
        file_path = f"mjd_data/{hdf5_file}"

        print("=" * 70)
        print(f"DOWNLOADING {split_name.upper()} FILE {file_idx}: {hdf5_file}")
        print("=" * 70)

        print(f"Ensuring full download: {hdf5_file}")
        os.system(f"wget -c --progress=bar:force -P mjd_data {url}")


        if os.path.exists(file_path):
            print(f"Download verified: {hdf5_file}\n")
        else:
            raise RuntimeError(f"Download failed: {hdf5_file}")

In [6]:
def sanity_report(name, values):
    values = np.array(values, dtype=float)

    nan_count  = np.isnan(values).sum()
    posinf_cnt = np.isposinf(values).sum()
    neginf_cnt = np.isneginf(values).sum()

    finite_vals = values[np.isfinite(values)]

    if finite_vals.size > 0:
        vmin = finite_vals.min()
        vmax = finite_vals.max()
        vmean = finite_vals.mean()
    else:
        vmin = vmax = vmean = np.nan

    print(
        f"{name:<20} | "
        f"NaN: {nan_count:<6} | "
        f"+Inf: {posinf_cnt:<4} | "
        f"-Inf: {neginf_cnt:<4} | "
        f"Min: {vmin:>10.4f} | "
        f"Max: {vmax:>10.4f} | "
        f"Mean: {vmean:>10.4f}"
    )

In [7]:
def extract_features(
    split_name,
    file_indices,
    output_folder,
    data_path,
    features=None,
    overwrite=False,
):
    """
    Resume-safe feature extraction:
    - If ID exists AND feature value is finite -> skip
    - If ID does not exist OR value is NaN -> compute & append

    Supports split_name: 'train', 'test', 'npml'
    Files:
      train -> MJD_Train_{i}.hdf5
      test  -> MJD_Test_{i}.hdf5
      npml  -> MJD_NPML_{i}.hdf5
    """
    os.makedirs(output_folder, exist_ok=True)

    # normalize split name
    split_key = str(split_name).strip().lower()
    split_to_prefix = {
        "train": "MJD_Train",
        "test":  "MJD_Test",
        "npml":  "MJD_NPML",
    }
    if split_key not in split_to_prefix:
        raise ValueError(
            f"Unknown split_name={split_name!r}. Use one of: {list(split_to_prefix.keys())}"
        )

    feature_registry = {
        "spectral_centroid_power": {"func": lambda wf, tp0: compute_spectral_centroid_power(wf)},
        "total_power":             {"func": lambda wf, tp0: compute_total_power(wf)},
        "current_kurtosis":        {"func": lambda wf, tp0: compute_current_kurtosis(wf, tp0)},
        "current_skewness":        {"func": lambda wf, tp0: compute_current_skewness(wf, tp0)},
        "tail_slope":              {"func": lambda wf, tp0: compute_tail_slope(wf, tp0)},
        "tail_slope_no_pz":        {"func": lambda wf, tp0: compute_tail_slope(wf, tp0, use_pz=False)},
        "time_to_peak":            {"func": lambda wf, tp0: compute_time_to_peak(wf, tp0)},
        "time_to_main_peak":       {"func": lambda wf, tp0: compute_time_to_main_peak(wf, tp0)},
        "current_width":           {"func": lambda wf, tp0: compute_current_width(wf, tp0)},
    }

    # -------- choose features --------
    if features is None:
        selected = list(feature_registry.keys())
    else:
        if isinstance(features, str):
            features = [features]
        selected = list(features)

    out_paths = {
        feat: os.path.join(output_folder, f"{feat}_{split_key}.csv")
        for feat in selected
    }

    # -------- overwrite option --------
    if overwrite:
        print("⚠️ Overwrite enabled — deleting existing CSVs")
        for path in out_paths.values():
            if os.path.exists(path):
                os.remove(path)
                print(f"  deleted {path}")
        print()

    # -------- preload existing IDs per feature --------
    existing = {}
    needs_fill = {}

    for feat, path in out_paths.items():
        if os.path.exists(path):
            df = pd.read_csv(path)
            existing[feat] = set(df["id"])
            needs_fill[feat] = set(df.loc[~np.isfinite(df[feat]), "id"])
        else:
            existing[feat] = set()
            needs_fill[feat] = set()

    # -------- main loop --------
    prefix = split_to_prefix[split_key]

    for file_idx in file_indices:
        hdf5_file = f"{prefix}_{file_idx}.hdf5"
        file_path = os.path.join(data_path, hdf5_file)

        print("=" * 70)
        print(f"PROCESSING {split_key.upper()} FILE {file_idx}: {hdf5_file}")
        print("=" * 70)

        data = load_hdf5(file_path)

        waveforms = data["raw_waveform"]
        tp0s      = data["tp0"]
        wf_ids    = data["id"]

        for i in range(len(waveforms)):
            wf = waveforms[i]
            tp0 = tp0s[i]
            full_id = f"{wf_ids[i]}_{split_key}_{file_idx}"

            for feat in selected:
                # skip if already exists AND has a finite value
                if full_id in existing[feat] and full_id not in needs_fill[feat]:
                    continue

                val = feature_registry[feat]["func"](wf, tp0)

                df_new = pd.DataFrame({"id": [full_id], feat: [val]})
                out_path = out_paths[feat]

                if not os.path.exists(out_path):
                    df_new.to_csv(out_path, index=False)
                else:
                    df_new.to_csv(out_path, mode="a", index=False, header=False)

                existing[feat].add(full_id)
                if not np.isfinite(val):
                    needs_fill[feat].add(full_id)
                elif full_id in needs_fill[feat]:
                    needs_fill[feat].remove(full_id)

        # ---- sanity checks output in the exact style you want ----
        print("Running sanity checks...")
        for feat in selected:
            df = pd.read_csv(out_paths[feat])

            # only rows from this file
            mask = df["id"].str.endswith(f"_{split_key}_{file_idx}")
            values = df.loc[mask, feat].values

            sanity_report(feat, values)
        print()

        print(f"Finished file {hdf5_file}\n")

    print(f"\n✅ Finished extracting features for split: {split_key}")

In [8]:
def extract_labels(
    split_name,
    file_indices,
    output_csv,
    data_folder="mjd_data",
    overwrite=False,
    columns=None  # None = all columns
):
    # Available fields in HDF5
    available_cols = [
        "energy_label",
        "psd_label_low_avse",
        "psd_label_high_avse",
        "psd_label_dcr",
        "psd_label_lq",
        "tp0",
        "id",
    ]

    # Resolve selected columns
    selected_cols = available_cols if columns is None else list(columns)

    # Basic progress prints
    print(f"\nStarting label extraction for split: {split_name}")
    print(f"Output CSV: {output_csv}")
    print(f"Selected columns: {selected_cols}")
    print(f"Number of files to process: {len(file_indices)}\n")

    # Overwrite existing file if requested
    if overwrite and os.path.exists(output_csv):
        os.remove(output_csv)
        print(f"Overwriting existing file: {output_csv}")

    os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)

    new_rows = []

    for i, file_idx in enumerate(file_indices, start=1):
        # Support train / test / npml
        if split_name == "train":
            hdf5_file = f"MJD_Train_{file_idx}.hdf5"
        elif split_name == "test":
            hdf5_file = f"MJD_Test_{file_idx}.hdf5"
        elif split_name == "npml":
            hdf5_file = f"MJD_NPML_{file_idx}.hdf5"   # change this if your actual filename pattern differs
        else:
            raise ValueError("split_name must be 'train', 'test', or 'npml'")

        h5_path = os.path.join(data_folder, hdf5_file)

        print(f"[{i}/{len(file_indices)}] Processing: {h5_path}")

        if not os.path.exists(h5_path):
            print("   Missing file, skipping.")
            continue

        with h5py.File(h5_path, "r") as file:
            raw_ids = np.array(file["id"])
            n = len(raw_ids)

            data = {}
            for col in selected_cols:
                if col == "id":
                    data["id"] = [f"{raw_ids[j]}_{split_name}_{file_idx}" for j in range(n)]
                else:
                    data[col] = np.array(file[col])

            df_new = pd.DataFrame(data)
            new_rows.append(df_new)
            print(f"   Done ({n} rows)")

    if len(new_rows) == 0:
        print("No files processed.")
        return

    df_all = pd.concat(new_rows, ignore_index=True)

    # Basic dedup if id exists
    if "id" in df_all.columns:
        before = len(df_all)
        df_all = df_all.drop_duplicates(subset="id", keep="last")
        after = len(df_all)
        print(f"\nDeduplicated by id: {before} -> {after} rows")

    df_all.to_csv(output_csv, index=False)
    print(f"\nSaved to {output_csv}")
    print("Finished label extraction.\n")

In [ ]:
# def extract_labels(
#     split_name,
#     file_indices,
#     output_csv,
#     data_folder = "mjd_data",
#     overwrite=False
# ):
#     """
#     Extract labels + raw waveform from multiple HDF5 files
#     and save/append into a single CSV.

#     ID format: <row_idx>_<split_name>_<file_idx>
    
#     Parameters
#     ----------
#     overwrite : bool
#         If True, remove existing CSV and start fresh.
#         If False, append to existing CSV and replace rows for current file_indices.
#     """

#     # Remove existing CSV if overwrite=True
#     if overwrite and os.path.exists(output_csv):
#         os.remove(output_csv)
#         print(f"🗑 Overwriting existing CSV: {output_csv}")

#     # Ensure folder exists
#     os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)

#     print(f"\nStarting label extraction for split: {split_name}")
#     print(f"Output CSV: {output_csv}")
#     print(f"Number of files to process: {len(file_indices)}\n")

#     # Load existing CSV if it exists and is readable
#     if os.path.exists(output_csv):
#         try:
#             df_existing = pd.read_csv(output_csv)
#             df_existing["id"] = df_existing["id"].astype(str).str.strip()

#             # Remove rows corresponding to current file_indices
#             for file_idx in file_indices:
#                 pattern = f"_{split_name}_{file_idx}$"
#                 df_existing = df_existing[~df_existing["id"].str.endswith(pattern)]

#         except pd.errors.EmptyDataError:
#             # File exists but has no valid CSV content
#             df_existing = pd.DataFrame()
#     else:
#         df_existing = pd.DataFrame()

#     write_header = not os.path.exists(output_csv)  # header needed only if file does not exist

#     new_rows = []  # collect new rows before concatenating

#     for file_idx in file_indices:
#         if split_name == "train":
#             hdf5_file = f"MJD_Train_{file_idx}.hdf5"
#         else:
#             hdf5_file = f"MJD_Test_{file_idx}.hdf5"
            
#         h5_path = os.path.join(data_folder, hdf5_file)

#         if not os.path.exists(h5_path):
#             print(f"⚠️ Missing file: {h5_path}, skipping")
#             continue

#         with h5py.File(h5_path, "r") as file:

#             # Load labels & waveform
#             energy_label        = np.array(file["energy_label"])
#             psd_label_low_avse  = np.array(file["psd_label_low_avse"])
#             psd_label_high_avse = np.array(file["psd_label_high_avse"])
#             psd_label_dcr       = np.array(file["psd_label_dcr"])
#             psd_label_lq        = np.array(file["psd_label_lq"])
#             tp0                 = np.array(file["tp0"])
#             ids                 = np.array(file["id"])

#             n = len(energy_label)

#             # Safety checks
#             assert all(len(x) == n for x in [
#                 psd_label_low_avse,
#                 psd_label_high_avse,
#                 psd_label_dcr,
#                 psd_label_lq,
#                 tp0,
#                 ids
#             ])

#             # Generate IDs
#             all_ids = [f"{ids[i]}_{split_name}_{file_idx}" for i in range(n)]

#             # Convert waveforms to lists for CSV storage if needed
#             # waveforms = [wf.tolist() for wf in raw_waveform]

#             df_new = pd.DataFrame({
#                 "id": all_ids,
#                 "energy_label": energy_label,
#                 "psd_label_low_avse": psd_label_low_avse,
#                 "psd_label_high_avse": psd_label_high_avse,
#                 "psd_label_dcr": psd_label_dcr,
#                 "psd_label_lq": psd_label_lq,
#                 "tp0": tp0,
#                 # "raw_waveform": waveforms,
#             })

#             new_rows.append(df_new)
#             print(f"✔ Processed {h5_path} ({n} rows)")

#     # Combine existing rows + new rows
#     if new_rows:
#         df_all = pd.concat([df_existing] + new_rows, ignore_index=True)
#     else:
#         df_all = df_existing

#     # Deduplicate just in case, keep last
#     df_all = df_all.drop_duplicates(subset="id", keep="last").reset_index(drop=True)

#     # Save
#     df_all.to_csv(output_csv, index=False)

#     # Verification
#     if df_all["id"].is_unique:
#         print("✅ Verification passed: all IDs are unique")
#     else:
#         print("❌ Verification failed: duplicate IDs remain")

#     print(f"\n✅ Finished label extraction for split: {split_name}")


### extract features from NPLM

In [ ]:
extract_features(
    split_name="npml",
    file_indices=range(3),
    output_folder="/Users/nomin1/Desktop/capstone/npml_features",
    data_path="/Volumes/SSK Drive/capstone/mjd_data/npml",
    features=None,
)

PROCESSING NPML FILE 0: MJD_NPML_0.hdf5


/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in exp
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in multiply
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:69: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(


Running sanity checks...
spectral_centroid_power | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    90.7900 | Max:  6217.6437 | Mean:   113.6667
total_power          | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     4.0745 | Max:    10.7068 | Mean:     8.9215
current_kurtosis     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    -1.3993 | Max:    12.8877 | Mean:     8.3868
current_skewness     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    -0.8922 | Max:     3.6786 | Mean:     2.9590
tail_slope           | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -2558.8569 | Max: 45568.3551 | Mean:    10.7724
tail_slope_no_pz     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -15435.7424 | Max: 39515.9066 | Mean: -9787.4868
time_to_peak         | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    39.0000 | Max:  2810.0000 | Mean:   114.1238
time_to_main_peak    | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    14.0000 | Max:   396.0000 | Mean:   109.7974
current_width        | NaN: 0      | +Inf: 

/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in exp
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in multiply
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:69: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(


Running sanity checks...
spectral_centroid_power | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    92.1093 | Max: 10318.0336 | Mean:   115.5483
total_power          | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     4.0043 | Max:    10.6935 | Mean:     8.9135
current_kurtosis     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    -1.4584 | Max:    24.2072 | Mean:     8.3752
current_skewness     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    -1.0531 | Max:     4.8107 | Mean:     2.9557
tail_slope           | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -3799.9311 | Max: 44922.4561 | Mean:     9.5106
tail_slope_no_pz     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -12906.7005 | Max: 38238.1961 | Mean: -9777.9896
time_to_peak         | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:  -708.0000 | Max:  2874.0000 | Mean:   114.5230
time_to_main_peak    | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     8.0000 | Max:   391.0000 | Mean:   109.9985
current_width        | NaN: 0      | +Inf: 

/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in exp
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in multiply
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:69: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(


Running sanity checks...
spectral_centroid_power | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    92.6473 | Max: 10881.4731 | Mean:   115.0052
total_power          | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     4.0952 | Max:    10.6699 | Mean:     8.9153
current_kurtosis     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    -1.6392 | Max:    16.0927 | Mean:     8.3760
current_skewness     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    -0.9242 | Max:     3.9477 | Mean:     2.9565
tail_slope           | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -1786.7598 | Max: 28622.2819 | Mean:     9.5555
tail_slope_no_pz     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -13798.9672 | Max: 22977.6746 | Mean: -9783.1621
time_to_peak         | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    28.0000 | Max:  2668.0000 | Mean:   113.9844
time_to_main_peak    | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    39.0000 | Max:   396.0000 | Mean:   109.9426
current_width        | NaN: 0      | +Inf: 

## Extracting Features

In [9]:
with_pz = ["tail_slope", "current_width"]
no_pz = ["spectral_centroid_power", "total_power", "current_kurtosis", "current_skewness", "time_to_peak", "time_to_main_peak", "tail_slope_no_pz"]

In [10]:
extract_features(
    split_name="test",
    file_indices=range(6),
    output_folder="/Users/nomin1/Desktop/capstone/test_features",
    data_path="/Volumes/SSK Drive/capstone/mjd_data/test",
    features=no_pz,
)

PROCESSING TEST FILE 0: MJD_Test_0.hdf5
Running sanity checks...
spectral_centroid_power | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    91.6678 | Max:  8129.3608 | Mean:   115.0461
total_power          | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     4.1375 | Max:    10.7146 | Mean:     8.9149
current_kurtosis     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    -1.6116 | Max:    12.7983 | Mean:     8.3692
current_skewness     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    -1.0408 | Max:     3.6487 | Mean:     2.9549
time_to_peak         | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:  -881.0000 | Max:  2707.0000 | Mean:   114.7041
time_to_main_peak    | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    36.0000 | Max:   393.0000 | Mean:   110.0620
tail_slope_no_pz     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -13839.2464 | Max: 46140.9577 | Mean: -9779.6194

Finished file MJD_Test_0.hdf5

PROCESSING TEST FILE 1: MJD_Test_1.hdf5
Running sanity checks...
spectral_centroid_powe

In [11]:
extract_features(
    split_name="train",
    file_indices=range(16),
    output_folder="/Users/nomin1/Desktop/capstone/train_features",
    data_path="/Volumes/SSK Drive/capstone/mjd_data/train",
    features=no_pz
)

PROCESSING TRAIN FILE 0: MJD_Train_0.hdf5
Running sanity checks...
spectral_centroid_power | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    90.4363 | Max:  7966.0628 | Mean:   113.9325
total_power          | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     4.4421 | Max:    10.6937 | Mean:     8.9142
current_kurtosis     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    -1.7530 | Max:    12.8351 | Mean:     8.3779
current_skewness     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    -0.9950 | Max:     3.6194 | Mean:     2.9571
time_to_peak         | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:  -413.0000 | Max:  2506.0000 | Mean:   114.4398
time_to_main_peak    | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:    21.0000 | Max:   389.0000 | Mean:   109.5998
tail_slope_no_pz     | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -18384.6755 | Max: 46656.8482 | Mean: -9779.9487

Finished file MJD_Train_0.hdf5

PROCESSING TRAIN FILE 1: MJD_Train_1.hdf5
Running sanity checks...
spectral_centroid

In [12]:
extract_features(
    split_name="test",
    file_indices=range(6),
    output_folder="/Users/nomin1/Desktop/capstone/test_features",
    data_path="/Volumes/SSK Drive/capstone/mjd_data/test",
    features=with_pz
)

PROCESSING TEST FILE 0: MJD_Test_0.hdf5
Running sanity checks...
tail_slope           | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -1694.2433 | Max: 50768.9451 | Mean:    13.0561
current_width        | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     0.0000 | Max:     1.9900 | Mean:     0.1493

Finished file MJD_Test_0.hdf5

PROCESSING TEST FILE 1: MJD_Test_1.hdf5
Running sanity checks...
tail_slope           | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -2918.0631 | Max: 34074.6411 | Mean:    10.5572
current_width        | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     0.0000 | Max:     1.9900 | Mean:     0.1489

Finished file MJD_Test_1.hdf5

PROCESSING TEST FILE 2: MJD_Test_2.hdf5
Running sanity checks...
tail_slope           | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -7442.9389 | Max: 33853.9865 | Mean:     6.9501
current_width        | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     0.0000 | Max:     1.9900 | Mean:     0.1499

Finished file MJD_Test_2.hdf5

PROCESSING TE

In [14]:
extract_features(
    split_name="train",
    file_indices=range(16),
    output_folder="/Users/nomin1/Desktop/capstone/train_features",
    data_path="/Volumes/SSK Drive/capstone/mjd_data/train",
    features=with_pz,
    overwrite=True
)

⚠️ Overwrite enabled — deleting existing CSVs

PROCESSING TRAIN FILE 0: MJD_Train_0.hdf5


/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in exp
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in multiply
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:69: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(


Running sanity checks...
tail_slope           | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -2887.4503 | Max: 52470.5476 | Mean:    11.6698
current_width        | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     0.0000 | Max:     1.9900 | Mean:     0.1494

Finished file MJD_Train_0.hdf5

PROCESSING TRAIN FILE 1: MJD_Train_1.hdf5


/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in exp
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in multiply
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:69: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(


Running sanity checks...
tail_slope           | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -4124.5228 | Max: 34887.2042 | Mean:    11.1119
current_width        | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     0.0000 | Max:     1.9767 | Mean:     0.1496

Finished file MJD_Train_1.hdf5

PROCESSING TRAIN FILE 2: MJD_Train_2.hdf5


...

Running sanity checks...
tail_slope           | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -92368.9938 | Max: 52264.0021 | Mean:     9.5144
current_width        | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     0.0000 | Max:     1.9900 | Mean:     0.1505

Finished file MJD_Train_13.hdf5

PROCESSING TRAIN FILE 14: MJD_Train_14.hdf5


/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in exp
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in multiply
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:69: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(


Running sanity checks...
tail_slope           | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -2961.0879 | Max: 55036.3957 | Mean:    10.7702
current_width        | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     0.0000 | Max:     1.9656 | Mean:     0.1493

Finished file MJD_Train_14.hdf5

PROCESSING TRAIN FILE 15: MJD_Train_15.hdf5


/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in exp
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:29: RuntimeWarning: overflow encountered in multiply
  return a * np.exp(-t / tau1) + b * np.exp(-t / tau2)
/Users/nomin1/Desktop/capstone/Majorana-Neutrino-Hunt/src/utils/transforms.py:69: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(


Running sanity checks...
tail_slope           | NaN: 0      | +Inf: 0    | -Inf: 0    | Min: -2137.1673 | Max: 56751.5608 | Mean:    12.2512
current_width        | NaN: 0      | +Inf: 0    | -Inf: 0    | Min:     0.0000 | Max:     1.9900 | Mean:     0.1490

Finished file MJD_Train_15.hdf5


✅ Finished extracting features for split: train


## Extract labels

In [ ]:
extract_labels(
    split_name="train",
    file_indices=range(0,16),
    output_csv="labels_train.csv",
    data_folder = "/Volumes/SSK Drive/mjd_data/train"
)


Starting label extraction for split: train
Output CSV: labels_train.csv
Number of files to process: 16

✔ Processed /Volumes/SSK Drive/mjd_data/train/MJD_Train_0.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/train/MJD_Train_1.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/train/MJD_Train_2.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/train/MJD_Train_3.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/train/MJD_Train_4.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/train/MJD_Train_5.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/train/MJD_Train_6.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/train/MJD_Train_7.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/train/MJD_Train_8.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/train/MJD_Train_9.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/train/MJD_Train_10.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/train/MJD_Train_11

In [ ]:
df = pd.read_csv("labels_train.csv")
df.shape

(1040000, 7)

In [ ]:
df.head()

,id,energy_label,psd_label_low_avse,psd_label_high_avse,psd_label_dcr,psd_label_lq,tp0
0,0_train_0,582.364295,False,True,True,True,957
1,1_train_0,250.159995,False,True,True,True,948
2,2_train_0,1212.323954,False,True,False,True,965
3,3_train_0,240.878110,False,True,True,False,927
4,4_train_0,285.124189,False,True,True,False,958


In [ ]:
extract_labels(
    split_name="test",
    file_indices=range(6),
    output_csv="labels_test.csv",
    data_folder = "/Volumes/SSK Drive/mjd_data/test"
)


Starting label extraction for split: test
Output CSV: labels_test.csv
Number of files to process: 6

✔ Processed /Volumes/SSK Drive/mjd_data/test/MJD_Test_0.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/test/MJD_Test_1.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/test/MJD_Test_2.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/test/MJD_Test_3.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/test/MJD_Test_4.hdf5 (65000 rows)
✔ Processed /Volumes/SSK Drive/mjd_data/test/MJD_Test_5.hdf5 (65000 rows)
✅ Verification passed: all IDs are unique

✅ Finished label extraction for split: test


In [ ]:
df_t = pd.read_csv("labels_test.csv")
df_t.shape

(390000, 7)

In [ ]:
df_t.head()

,id,energy_label,psd_label_low_avse,psd_label_high_avse,psd_label_dcr,psd_label_lq,tp0
0,2395098_test_0,1167.174731,True,True,True,True,967
1,2395099_test_0,870.765543,False,True,True,False,960
2,2395100_test_0,582.980526,False,True,True,True,960
3,2395101_test_0,238.918902,True,True,True,True,930
4,2395102_test_0,214.491195,False,True,True,True,924


In [11]:
extract_labels(
    split_name="npml",
    file_indices=range(3),
    output_csv="npml_features/tp0_npml.csv",
    data_folder="/Volumes/SSK Drive/capstone/mjd_data/npml",
    overwrite=False,
    columns=["id", "tp0"]
)


Starting label extraction for split: npml
Output CSV: npml_features/tp0_npml.csv
Selected columns: ['id', 'tp0']
Number of files to process: 3

[1/3] Processing: /Volumes/SSK Drive/capstone/mjd_data/npml/MJD_NPML_0.hdf5
   Done (65000 rows)
[2/3] Processing: /Volumes/SSK Drive/capstone/mjd_data/npml/MJD_NPML_1.hdf5
   Done (65000 rows)
[3/3] Processing: /Volumes/SSK Drive/capstone/mjd_data/npml/MJD_NPML_2.hdf5
   Done (29697 rows)

Deduplicated by id: 159697 -> 159697 rows

Saved to npml_features/tp0_npml.csv
Finished label extraction.



## Merge csv files

### combine npml csv files

In [12]:
files_npml = glob.glob("npml_features/*npml.csv")
files_npml

['npml_features/tail_slope_no_pz_npml.csv',
 'npml_features/current_kurtosis_npml.csv',
 'npml_features/time_to_peak_npml.csv',
 'npml_features/total_power_npml.csv',
 'npml_features/tail_slope_npml.csv',
 'npml_features/current_skewness_npml.csv',
 'npml_features/time_to_main_peak_npml.csv',
 'npml_features/spectral_centroid_power_npml.csv',
 'npml_features/current_width_npml.csv',
 'npml_features/tp0_npml.csv']

In [13]:
dfs_npml = [pd.read_csv(f) for f in files_npml]
merged_npml = reduce(lambda l, r: pd.merge(l, r, on="id", how="inner"), dfs_npml)
merged_npml.head()

,id,tail_slope_no_pz,current_kurtosis,time_to_peak,total_power,tail_slope,current_skewness,time_to_main_peak,spectral_centroid_power,current_width,tp0
0,3033789_npml_0,-10061.013124,6.674751,83,8.638530,-4.313729,2.738811,83,104.055619,0.159431,968
1,3033790_npml_0,-9089.512080,10.563529,180,8.397907,-13.898763,3.335234,180,107.098676,0.121488,942
2,3033791_npml_0,-9449.901755,5.231941,185,8.444459,2.577768,2.477587,185,103.335168,0.182753,940
3,3033792_npml_0,-8567.556832,7.421443,201,8.378091,-0.236396,2.862397,201,106.077102,0.154580,936
4,3033793_npml_0,-10081.558061,8.552893,96,8.453313,3.093208,3.057418,96,109.909804,0.143348,952


In [14]:
merged_npml.shape

(159697, 11)

In [16]:
merged_npml.to_csv("npml_features/combined_npml_n.csv", index=False)
os.system('gzip -k npml_features/combined_npml_n.csv') # compress the file

0

### combining train csv files

In [15]:
files_train = glob.glob("train_features/*.csv")
files_train

['train_features/current_skewness_train.csv',
 'train_features/tail_slope_no_pz_train.csv',
 'train_features/spectral_centroid_power_train.csv',
 'train_features/current_kurtosis_train.csv',
 'train_features/tail_slope_train.csv',
 'train_features/total_power_train.csv',
 'train_features/time_to_main_peak_train.csv',
 'train_features/time_to_peak_train.csv',
 'train_features/current_width_train.csv']

In [16]:
dfs_train = [pd.read_csv(f) for f in files_train]
merged_train = reduce(lambda l, r: pd.merge(l, r, on="id", how="inner"), dfs_train)
merged_train.head()

,id,current_skewness,tail_slope_no_pz,spectral_centroid_power,current_kurtosis,tail_slope,total_power,time_to_main_peak,time_to_peak,current_width
0,0_train_0,3.138621,-10097.252452,107.276207,9.373445,1.975741,9.232819,85,85,0.126727
1,1_train_0,3.104545,-9398.326178,108.213621,8.950671,-1.501815,8.475871,87,87,0.136277
2,2_train_0,2.122413,-10322.700267,105.735183,3.296724,-3.050903,8.795490,95,95,0.236836
3,3_train_0,2.793886,-8370.010321,107.946935,6.655448,-1.657508,8.448951,116,116,0.171464
4,4_train_0,2.269455,-9623.731683,106.350372,3.838895,-3.879321,8.589440,94,94,0.225977


In [17]:
merged_train.shape

(1040000, 10)

In [18]:
merged_train.to_csv("train_features/combined_train_n.csv", index=False)

In [19]:
os.system('gzip -k train_features/combined_train_n.csv') # compress the file

0

### combining test csv files

In [20]:
files_test = glob.glob("test_features/*.csv")
files_test

['test_features/total_power_test.csv',
 'test_features/tail_slope_test.csv',
 'test_features/time_to_main_peak_test.csv',
 'test_features/current_skewness_test.csv',
 'test_features/tail_slope_no_pz_test.csv',
 'test_features/current_kurtosis_test.csv',
 'test_features/time_to_peak_test.csv',
 'test_features/spectral_centroid_power_test.csv',
 'test_features/current_width_test.csv']

In [21]:
dfs_test = [pd.read_csv(f) for f in files_test]
merged_test = reduce(lambda l, r: pd.merge(l, r, on="id", how="inner"), dfs_test)
merged_test.head()

,id,total_power,tail_slope,time_to_main_peak,current_skewness,tail_slope_no_pz,current_kurtosis,time_to_peak,spectral_centroid_power,current_width
0,2395098_test_0,9.818632,-0.841532,77,3.257910,-10545.834394,10.122620,77,108.796954,0.122269
1,2395099_test_0,9.556397,32.147544,99,3.377049,-10647.558850,10.875182,99,110.305348,0.118442
2,2395100_test_0,9.232249,0.558690,100,2.662094,-10110.162247,6.156408,100,108.213213,0.171536
3,2395101_test_0,8.453703,-5.024905,110,3.407008,-9936.410284,11.064097,110,109.196803,0.116946
4,2395102_test_0,7.291795,-1.082581,135,2.489377,-8275.481025,5.544923,135,109.163049,0.163692


In [22]:
merged_test.shape

(390000, 10)

In [23]:
merged_test.to_csv("test_features/combined_test_n.csv", index=False)

In [24]:
os.system('gzip -k test_features/combined_test_n.csv') # compress the file

0